# HNC Dementia + MDD experiments

Runs all three protocols on the private Dementia and MDD datasets:

1. **Frozen LaBraM features + η²** — measures subject-vs-label variance dominance.
2. **Classical ML baseline** — band-power features + RandomForest / LogisticRegression with subject-level 5-fold CV.
3. **LaBraM fine-tuning** — full fine-tuning with subject-level 5-fold CV (calls `train_ft.py`).

Run cells top-to-bottom. All outputs are written to `results/`. Run from the
repo root, with the `FM_analysis` env active. See `docs/setup.md` for
environment setup instructions (for IT) and the inline cells below for
data preparation (for the researcher).

## 0. Environment + data check

In [ ]:
import os, sys
import numpy as np
import torch
import pandas as pd

# Make sure we can import the repo modules regardless of where the kernel started
REPO_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print('repo root:', REPO_ROOT)
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('numpy', np.__version__, '| pandas', pd.__version__)

for f in [
    'data/hnc/Dementia_paper_dataset_rawData.hdf5',
    'data/hnc/Dementia_paper_dataset_info.pkl',
    'data/hnc/MDD_paper_dataset_rawData.hdf5',
    'data/hnc/MDD_paper_dataset_info.pkl',
    'weights/labram-base/pretrained_weights.pth',
]:
    print(f, 'OK' if os.path.exists(f) else '*** MISSING ***')

## 1. Build the datasets (one-time preprocessing + caching)

First call triggers HDF5 load → channel selection → resample 500→200 Hz → epoch into 5s windows → z-score → cache as `.pt`. Subsequent calls hit the cache and are instant. If the HDF5 isn't 19 channels, the second cell tells you what to pass for `channel_names=`.

In [ ]:
from pipeline.hnc_dataset import HNCDataset

# ──────────────────────────────────────────────────────────────────────────
# Channel names (REQUIRED) — paste the 30 channel names from the data owner
# in HDF5 axis-1 order. Same list works for both Dementia and MDD unless the
# data owner says otherwise.
#
# Until you set this, the loader will hard-fail with a clear error.
# ──────────────────────────────────────────────────────────────────────────
HNC_CHANNEL_NAMES = None  # e.g. ['Fp1','Fp2','F3','F4','F7','F8','Fz', ...]

assert HNC_CHANNEL_NAMES is not None, (
    'Set HNC_CHANNEL_NAMES to the 30-channel order from the data owner '
    'before running the rest of the notebook.'
)
assert len(HNC_CHANNEL_NAMES) == 30, (
    f'expected 30 channel names, got {len(HNC_CHANNEL_NAMES)}'
)

ds_dem = HNCDataset(name='dementia', data_root='data/hnc',
                    channel_names=HNC_CHANNEL_NAMES,
                    target_sfreq=200.0, window_sec=5.0, norm='zscore', binary=True)
ds_mdd = HNCDataset(name='mdd', data_root='data/hnc',
                    channel_names=HNC_CHANNEL_NAMES,
                    target_sfreq=200.0, window_sec=5.0, norm='zscore', binary=True)

for tag, ds in [('dementia', ds_dem), ('mdd', ds_mdd)]:
    epochs, label, n_epochs, sid = ds[0]
    print(f'{tag}: item0 shape={tuple(epochs.shape)} label={label} n_epochs={n_epochs}')

## 2. Frozen LaBraM features + η²

Loads LaBraM pretrained weights, extracts mean-pooled features per recording, computes
$\eta^2_{\text{subject}}$ and $\eta^2_{\text{label}}$, and reports the ratio. A high
ratio means subject identity dominates the embedding (the central finding from our
cross-dataset analysis on UCSD Stress / ADFTD / TDBRAIN).

In [ ]:
import baseline.labram  # noqa: F401  (registers the extractor)
from baseline.abstract.factory import create_extractor
from baseline.labram.channel_map import get_input_chans
from pipeline.common_channels import COMMON_19

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

def build_labram():
    extractor = create_extractor('labram')
    extractor.input_chans = get_input_chans(COMMON_19)
    extractor.eval().to(DEVICE)
    return extractor

@torch.no_grad()
def extract_pooled_features(ds, extractor, batch_size=16):
    feats, labels, pids = [], [], []
    for i in range(len(ds)):
        epochs, label, n_epochs, sid = ds[i]
        chunks = []
        for s in range(0, epochs.shape[0], batch_size):
            batch = epochs[s:s+batch_size].to(DEVICE)
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE.startswith('cuda')):
                f = extractor(batch).float().cpu()
            chunks.append(f)
        pooled = torch.cat(chunks, dim=0).mean(dim=0).numpy()
        feats.append(pooled)
        labels.append(label)
        pids.append(sid)
        if (i + 1) % 50 == 0 or i == 0:
            print(f'  [{i+1}/{len(ds)}] epochs={epochs.shape[0]}, pooled={pooled.shape}')
    return (np.stack(feats), np.array(labels), np.array(pids))

def eta_squared(features: np.ndarray, factor: np.ndarray) -> float:
    f = np.asarray(features, dtype=np.float64)
    g = np.asarray(factor)
    grand_mean = f.mean(axis=0, keepdims=True)
    ss_total = ((f - grand_mean) ** 2).sum(axis=0)
    ss_between = np.zeros(f.shape[1])
    for u in np.unique(g):
        mask = g == u
        if mask.sum() < 2:
            continue
        gmean = f[mask].mean(axis=0)
        ss_between += mask.sum() * (gmean - grand_mean.squeeze()) ** 2
    return float(np.mean(ss_between / (ss_total + 1e-12)))

extractor = build_labram()
print('LaBraM loaded\n')

FROZEN_DIR = 'results/cross_dataset'
os.makedirs(FROZEN_DIR, exist_ok=True)

frozen_summary = []
for tag, ds in [('dementia', ds_dem), ('mdd', ds_mdd)]:
    print(f'== {tag} ==')
    feats, labels, pids = extract_pooled_features(ds, extractor)
    out = f'{FROZEN_DIR}/features_hnc_{tag}_19ch.npz'
    np.savez_compressed(out, features=feats, labels=labels, patient_ids=pids)
    eta_subj  = eta_squared(feats, pids)
    eta_label = eta_squared(feats, labels)
    ratio = eta_subj / max(eta_label, 1e-9)
    print(f'  saved {out} | features={feats.shape} subjects={len(np.unique(pids))}')
    print(f'  eta_subject = {eta_subj:.4f}')
    print(f'  eta_label   = {eta_label:.4f}')
    print(f'  ratio       = {ratio:.2f}x  (>1 means subject identity dominates)')
    print()
    frozen_summary.append({
        'dataset': tag,
        'n_recordings': len(ds),
        'n_subjects': int(len(np.unique(pids))),
        'eta_subject_frozen': eta_subj,
        'eta_label_frozen':   eta_label,
        'ratio_frozen':       ratio,
    })

import json
with open(f'{FROZEN_DIR}/hnc_frozen_summary.json', 'w') as f:
    json.dump(frozen_summary, f, indent=2)
frozen_summary

## 3. Classical ML baseline (band power + RandomForest / LogReg)

Per-recording mean band power (theta / alpha / beta) across 19 channels → 57-dim feature vector → 5-fold StratifiedGroupKFold (subject-level).

In [ ]:
from scipy.signal import welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    cohen_kappa_score, f1_score,
)

SFREQ = 200.0
BANDS = {'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30)}

def bandpower_features(epochs_tensor):
    """Average per-recording band power. Input: (M, 19, T) tensor; output: (57,) array."""
    arr = epochs_tensor.numpy()  # (M, 19, T)
    pooled = arr.mean(axis=0)    # (19, T) — average across epochs first
    feats = []
    for ch in range(pooled.shape[0]):
        freqs, psd = welch(pooled[ch], fs=SFREQ, nperseg=min(512, pooled.shape[1]))
        for lo, hi in BANDS.values():
            band_mask = (freqs >= lo) & (freqs < hi)
            feats.append(psd[band_mask].mean() if band_mask.any() else 0.0)
    return np.asarray(feats, dtype=np.float64)

def build_band_features(ds):
    X, y, g = [], [], []
    for i in range(len(ds)):
        epochs, label, _n, sid = ds[i]
        X.append(bandpower_features(epochs))
        y.append(label)
        g.append(sid)
        if (i + 1) % 100 == 0 or i == 0:
            print(f'  [{i+1}/{len(ds)}]')
    return np.stack(X), np.array(y), np.array(g)

def cv_score(X, y, g, model, n_splits=5, seed=42):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics = {'acc': [], 'bal_acc': [], 'f1': [], 'kappa': []}
    for fold, (tr, te) in enumerate(cv.split(X, y, g)):
        clf = Pipeline([('scale', StandardScaler()), ('clf', model)])
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        metrics['acc'].append(accuracy_score(y[te], pred))
        metrics['bal_acc'].append(balanced_accuracy_score(y[te], pred))
        metrics['f1'].append(f1_score(y[te], pred, average='binary'))
        metrics['kappa'].append(cohen_kappa_score(y[te], pred))
    return {k: (float(np.mean(v)), float(np.std(v))) for k, v in metrics.items()}

classical_summary = []
for tag, ds in [('dementia', ds_dem), ('mdd', ds_mdd)]:
    print(f'== {tag} ==')
    X, y, g = build_band_features(ds)
    print(f'  features={X.shape}  pos={int(y.sum())}/{len(y)}  subjects={len(np.unique(g))}')
    rf = cv_score(X, y, g, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
    lr = cv_score(X, y, g, LogisticRegression(max_iter=2000, C=1.0, random_state=42))
    print(f'  RF     bal_acc={rf["bal_acc"][0]:.3f}±{rf["bal_acc"][1]:.3f}  f1={rf["f1"][0]:.3f}')
    print(f'  LogReg bal_acc={lr["bal_acc"][0]:.3f}±{lr["bal_acc"][1]:.3f}  f1={lr["f1"][0]:.3f}')
    print()
    classical_summary.append({'dataset': tag, 'RF': rf, 'LogReg': lr,
                              'n_features': X.shape[1], 'n_recordings': len(y),
                              'n_subjects': int(len(np.unique(g)))})

with open('results/cross_dataset/hnc_classical_summary.json', 'w') as f:
    json.dump(classical_summary, f, indent=2)
classical_summary

## 4. LaBraM fine-tuning (subject-level 5-fold CV)

Calls `train_ft.py` as a subprocess for each dataset. This is the slow one — expect ~30–60 minutes per dataset on a 4090, depending on the number of subjects. Results land under `results/<timestamp>_ft_*/`.

If you want to skip fine-tuning for a quick first pass, comment out the `!python` lines.

In [ ]:
# Free GPU memory before launching the subprocess (optional but helpful)
del extractor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Pass the channel list as a comma-separated string to train_ft.py
hnc_chs = ','.join(HNC_CHANNEL_NAMES)

# Dementia — full FT, subject-level 5-fold CV
# train_ft.py run_id pattern: {date}_{time}_ft_hncdementia_labram_dementia/
!python train_ft.py --mode ft --dataset dementia --extractor labram \
    --hnc-data-root data/hnc --hnc-channels "$hnc_chs" --device cuda:0 \
    --folds 5 --epochs 30 --batch-size 16 --lr 5e-5 --weight-decay 0.05 \
    --norm zscore

# MDD — full FT, subject-level 5-fold CV
# train_ft.py run_id pattern: {date}_{time}_ft_hncmdd_labram_mdd/
!python train_ft.py --mode ft --dataset mdd --extractor labram \
    --hnc-data-root data/hnc --hnc-channels "$hnc_chs" --device cuda:0 \
    --folds 5 --epochs 30 --batch-size 16 --lr 5e-5 --weight-decay 0.05 \
    --norm zscore

## 5. Collect results to download

Bundle up everything that should come back from the server.

In [ ]:
import shutil, glob, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
out_dir = f'results/hnc_bundle_{stamp}'
os.makedirs(out_dir, exist_ok=True)

wanted = []
wanted += glob.glob('results/cross_dataset/features_hnc_*_19ch.npz')
wanted += glob.glob('results/cross_dataset/hnc_frozen_summary.json')
wanted += glob.glob('results/cross_dataset/hnc_classical_summary.json')
# train_ft.py run_id format: {YYYYMMDD}_{HHMM}_{mode}_{label}_{extractor}_{dataset}
# label_tag is args.label.replace("-", ""), so "hnc-dementia" → "hncdementia"
for pat in [
    'results/*_ft_hncdementia_labram_dementia/summary.json',
    'results/*_ft_hncdementia_labram_dementia/curves_fold*.csv',
    'results/*_ft_hncdementia_labram_dementia/config.json',
    'results/*_ft_hncmdd_labram_mdd/summary.json',
    'results/*_ft_hncmdd_labram_mdd/curves_fold*.csv',
    'results/*_ft_hncmdd_labram_mdd/config.json',
]:
    wanted += glob.glob(pat)

for src in wanted:
    rel = os.path.relpath(src, 'results')
    dst = os.path.join(out_dir, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)
shutil.make_archive(out_dir, 'zip', out_dir)
print('bundle ready:', out_dir + '.zip')
print(f'{len(wanted)} files:')
for w in wanted:
    print(' ', w)